In [1]:
# ============================================================
# Section 1. Function definitions
# Transfer matrix for disordered QWZ model
# ============================================================

import os
import csv
import time
import numpy as np
import pandas as pd
import scipy.linalg as la
import matplotlib.pyplot as plt


# -----------------------------
# Pauli matrices
# -----------------------------

sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
id2 = np.eye(2, dtype=complex)


# -----------------------------
# QWZ hopping blocks
# -----------------------------

def qwz_hopping_blocks(A=1.5, t=1.0):
    """
    QWZ convention:

    H(k) = -A sin(kx) sx - A sin(ky) sy
           + [u + t cos(kx) + t cos(ky)] sz

    Real-space hopping:
        H_{r,r+x} = Tx
        H_{r,r+y} = Ty
    """
    Tx = 0.5 * t * sz + 0.5j * A * sx
    Ty = 0.5 * t * sz + 0.5j * A * sy
    return Tx, Ty


def build_slice_hamiltonian(M, impurity_mask, u=-2.05, A=1.5, t=1.0, h=-1.0):
    """
    Build the Hamiltonian H_x inside one transverse slice.

    Slice width: M
    Orbitals per site: 2
    Slice dimension: 2M

    impurity_mask[y] = True means site (x,y) has impurity.
    """
    dim = 2 * M
    Hx = np.zeros((dim, dim), dtype=complex)

    _, Ty = qwz_hopping_blocks(A=A, t=t)

    for y in range(M):
        sl = slice(2*y, 2*y + 2)

        onsite = u * sz
        if impurity_mask[y]:
            onsite = onsite - h * sz

        Hx[sl, sl] += onsite

        # periodic transverse hopping y -> y+1
        yp = (y + 1) % M
        slp = slice(2*yp, 2*yp + 2)

        Hx[sl, slp] += Ty
        Hx[slp, sl] += Ty.conj().T

    return Hx


def random_impurity_mask(M, nd=0.03, rng=None):
    """
    Bernoulli disorder for one slice.
    Each transverse site has impurity probability nd.
    """
    if rng is None:
        rng = np.random.default_rng()
    return rng.random(M) < nd


def build_transfer_matrix_for_slice(
    M,
    impurity_mask,
    E=0.0,
    u=-2.05,
    A=1.5,
    t=1.0,
    h=-1.0,
):
    """
    Build transfer matrix T_x such that:

        [psi_{x+1}]   [Vx^{-1}(E-Hx), -Vx^{-1}Vx^dag] [psi_x]
        [psi_x    ] = [I,               0             ] [psi_{x-1}]

    where Vx = I_M kron Tx.
    """
    Tx, _ = qwz_hopping_blocks(A=A, t=t)
    Hx = build_slice_hamiltonian(M, impurity_mask, u=u, A=A, t=t, h=h)

    dim = 2 * M
    I = np.eye(dim, dtype=complex)

    Vx = np.kron(np.eye(M, dtype=complex), Tx)

    Ablock = la.solve(Vx, E * I - Hx)
    Bblock = -la.solve(Vx, Vx.conj().T)

    upper = np.hstack([Ablock, Bblock])
    lower = np.hstack([I, np.zeros_like(I)])

    return np.vstack([upper, lower])


# -----------------------------
# Lyapunov exponents
# -----------------------------

def lyapunov_exponents_qr(
    M,
    h,
    n_slices=100000,
    E=0.0,
    u=-2.05,
    A=1.5,
    t=1.0,
    nd=0.03,
    seed=12345,
    qr_interval=1,
    verbose=False,
):
    """
    Compute Lyapunov exponents by QR stabilization.

    Output:
        gamma_min_pos
        xi_M = 1/gamma_min_pos
        Lambda = xi_M/M
    """
    rng = np.random.default_rng(seed)

    dim_slice = 2 * M
    dim_T = 2 * dim_slice

    Q = np.eye(dim_T, dtype=complex)
    log_diag_accum = np.zeros(dim_T, dtype=float)

    n_qr = 0
    t0 = time.time()

    for x in range(n_slices):
        impurity_mask = random_impurity_mask(M, nd=nd, rng=rng)

        Tmat = build_transfer_matrix_for_slice(
            M=M,
            impurity_mask=impurity_mask,
            E=E,
            u=u,
            A=A,
            t=t,
            h=h,
        )

        Q = Tmat @ Q

        if (x + 1) % qr_interval == 0:
            Q, R = la.qr(Q, mode="economic")
            diag_R = np.abs(np.diag(R))
            diag_R = np.maximum(diag_R, 1e-300)
            log_diag_accum += np.log(diag_R)
            n_qr += 1

        if verbose and (x + 1) % max(1, n_slices // 5) == 0:
            print(f"  slice {x+1}/{n_slices}")

    total_length = n_qr * qr_interval
    gammas = np.sort(np.real(log_diag_accum / total_length))

    positive = gammas[gammas > 1e-10]

    if len(positive) == 0:
        gamma_min_pos = np.nan
        xi_M = np.inf
        Lambda = np.inf
    else:
        gamma_min_pos = positive[0]
        xi_M = 1.0 / gamma_min_pos
        Lambda = xi_M / M

    runtime = time.time() - t0

    return {
        "M": M,
        "h": h,
        "n_slices": n_slices,
        "seed": seed,
        "gamma_min_pos": gamma_min_pos,
        "xi_M": xi_M,
        "Lambda": Lambda,
        "gammas": gammas,
        "runtime_sec": runtime,
    }


# -----------------------------
# Scan driver with checkpointing
# -----------------------------

def transfer_scan(
    M_list,
    h_values,
    n_slices=100000,
    n_seeds=4,
    seed0=202607200,
    out_csv="tm_scan_raw.csv",
    model_params=None,
):
    """
    Run transfer-matrix scan with checkpointing.

    Saves one row per (M,h,seed).
    """
    if model_params is None:
        model_params = {
            "u": -2.05,
            "A": 1.5,
            "t": 1.0,
            "nd": 0.03,
            "E": 0.0,
        }

    fields = [
        "M", "h", "n_slices", "seed", "seed_index",
        "u", "A", "t", "nd", "E",
        "gamma_min_pos", "xi_M", "Lambda", "runtime_sec",
    ]

    done = set()
    if os.path.exists(out_csv):
        old = pd.read_csv(out_csv)
        for _, r in old.iterrows():
            done.add((int(r["M"]), float(r["h"]), int(r["seed"])))

    def append_row(row):
        file_exists = os.path.exists(out_csv)
        with open(out_csv, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            if not file_exists:
                writer.writeheader()
            writer.writerow(row)

    for M in M_list:
        for ih, h in enumerate(h_values):
            for iseed in range(n_seeds):
                seed = int(seed0 + 1000000*M + 1000*ih + iseed)
                key = (int(M), float(h), int(seed))

                if key in done:
                    print(f"[skip] M={M}, h={h:.6f}, seed={seed}")
                    continue

                print("\n" + "="*80)
                print(
                    f"[run] M={M}, h={h:.6f}, "
                    f"seed {iseed+1}/{n_seeds}, "
                    f"n_slices={n_slices}, seed={seed}"
                )
                print("="*80)

                res = lyapunov_exponents_qr(
                    M=M,
                    h=float(h),
                    n_slices=n_slices,
                    E=model_params["E"],
                    u=model_params["u"],
                    A=model_params["A"],
                    t=model_params["t"],
                    nd=model_params["nd"],
                    seed=seed,
                    qr_interval=1,
                    verbose=False,
                )

                row = {
                    "M": int(M),
                    "h": float(h),
                    "n_slices": int(n_slices),
                    "seed": int(seed),
                    "seed_index": int(iseed),
                    "u": model_params["u"],
                    "A": model_params["A"],
                    "t": model_params["t"],
                    "nd": model_params["nd"],
                    "E": model_params["E"],
                    "gamma_min_pos": float(res["gamma_min_pos"]),
                    "xi_M": float(res["xi_M"]),
                    "Lambda": float(res["Lambda"]),
                    "runtime_sec": float(res["runtime_sec"]),
                }

                append_row(row)

                print(
                    f"gamma_min={res['gamma_min_pos']:.6e}, "
                    f"xi={res['xi_M']:.3f}, "
                    f"Lambda={res['Lambda']:.3f}, "
                    f"time={res['runtime_sec']:.1f}s"
                )

    raw = pd.read_csv(out_csv)

    summary = raw.groupby(["M", "h"], as_index=False).agg(
        n=("Lambda", "count"),
        mean_gamma=("gamma_min_pos", "mean"),
        std_gamma=("gamma_min_pos", "std"),
        mean_xi=("xi_M", "mean"),
        std_xi=("xi_M", "std"),
        mean_Lambda=("Lambda", "mean"),
        std_Lambda=("Lambda", "std"),
        median_Lambda=("Lambda", "median"),
        min_Lambda=("Lambda", "min"),
        max_Lambda=("Lambda", "max"),
        mean_runtime_sec=("runtime_sec", "mean"),
    )

    summary_csv = out_csv.replace("_raw.csv", "_summary.csv")
    summary.to_csv(summary_csv, index=False)

    print("\nSaved raw:", out_csv)
    print("Saved summary:", summary_csv)

    return raw, summary

# Setup and Conduct

In [ ]:
# ============================================================
# Section 2. Simulation setup and conduct
# Left-boundary transfer-matrix scan
# ============================================================

model_params = {
    "u": -2.05,
    "A": 1.5,
    "t": 1.0,
    "nd": 0.03,
    "E": 0.0,
}

# Left impurity-bound-state boundary from Bott index:
# h_c roughly around -3.45

# Option A: clean fine scan for M=8,12,16
M_list_main = [8, 12, 16]
h_values_main = np.round(np.arange(-3.458, -3.442 + 1e-12, 0.002), 6)

raw_main, summary_main = transfer_scan(
    M_list=M_list_main,
    h_values=h_values_main,
    n_slices=100000,
    n_seeds=4,
    seed0=202607200,
    out_csv="tm_left_clean_main_raw.csv",
    model_params=model_params,
)

display(summary_main)


# Option B: focused larger-M scan.
# Run this after Option A finishes.
# You may comment it out if you only want the first scan.

M_list_large = [20, 24]
h_values_large = np.array([-3.454, -3.452, -3.450, -3.448, -3.446])

raw_large, summary_large = transfer_scan(
    M_list=M_list_large,
    h_values=h_values_large,
    n_slices=100000,
    n_seeds=3,
    seed0=202607210,
    out_csv="tm_left_clean_largeM_raw.csv",
    model_params=model_params,
)

display(summary_large)


# Combine results
raw_all = pd.concat([raw_main, raw_large], ignore_index=True)
summary_all = pd.concat([summary_main, summary_large], ignore_index=True)

raw_all.to_csv("tm_left_clean_combined_raw.csv", index=False)
summary_all.to_csv("tm_left_clean_combined_summary.csv", index=False)

display(summary_all.sort_values(["M", "h"]))


[run] M=8, h=-3.458000, seed 1/4, n_slices=100000, seed=210607200
gamma_min=8.596779e-03, xi=116.323, Lambda=14.540, time=39.9s

[run] M=8, h=-3.458000, seed 2/4, n_slices=100000, seed=210607201
gamma_min=8.060439e-03, xi=124.063, Lambda=15.508, time=39.9s

[run] M=8, h=-3.458000, seed 3/4, n_slices=100000, seed=210607202
gamma_min=9.134166e-03, xi=109.479, Lambda=13.685, time=40.1s

[run] M=8, h=-3.458000, seed 4/4, n_slices=100000, seed=210607203


# Plotting and Analyzing

In [ ]:
# ============================================================
# Section 3. Plotting and analyzing
# ============================================================

def plot_transfer_mean(summary, title="Transfer-matrix scan: mean Lambda"):
    plt.figure(figsize=(7.2, 4.8), dpi=150)

    for M, ss in summary.groupby("M"):
        ss = ss.sort_values("h")
        yerr = ss["std_Lambda"].values
        yerr = np.where(np.isnan(yerr), 0.0, yerr)

        plt.errorbar(
            ss["h"],
            ss["mean_Lambda"],
            yerr=yerr,
            marker="o",
            capsize=3,
            linewidth=1.5,
            label=f"M={M}",
        )

    plt.xlabel("h")
    plt.ylabel(r"$\Lambda_M=\xi_M/M$")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def plot_transfer_median(summary, title="Transfer-matrix scan: median Lambda"):
    plt.figure(figsize=(7.2, 4.8), dpi=150)

    for M, ss in summary.groupby("M"):
        ss = ss.sort_values("h")

        plt.plot(
            ss["h"],
            ss["median_Lambda"],
            marker="o",
            linewidth=1.5,
            label=f"M={M}",
        )

    plt.xlabel("h")
    plt.ylabel(r"median $\Lambda_M$")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def plot_seed_scatter(raw, title="Transfer-matrix scan: individual seeds"):
    plt.figure(figsize=(7.2, 4.8), dpi=150)

    for M, ss in raw.groupby("M"):
        ss = ss.sort_values("h")
        plt.scatter(
            ss["h"],
            ss["Lambda"],
            s=24,
            alpha=0.65,
            label=f"M={M}",
        )

    plt.xlabel("h")
    plt.ylabel(r"$\Lambda_M=\xi_M/M$")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def plot_trial_collapse(summary, hc=-3.450, nu=2.4, use_median=False):
    """
    Diagnostic collapse plot:
        x = (h - hc) M^{1/nu}

    This is not an automatic fit.
    Use it only to visually test candidate hc, nu.
    """
    plt.figure(figsize=(7.2, 4.8), dpi=150)

    ycol = "median_Lambda" if use_median else "mean_Lambda"
    ylabel = r"median $\Lambda_M$" if use_median else r"mean $\Lambda_M$"

    for M, ss in summary.groupby("M"):
        ss = ss.sort_values("h")
        x = (ss["h"].values - hc) * (M ** (1.0 / nu))
        y = ss[ycol].values

        plt.plot(
            x,
            y,
            marker="o",
            linewidth=1.5,
            label=f"M={M}",
        )

    plt.xlabel(r"$(h-h_c)M^{1/\nu}$")
    plt.ylabel(ylabel)
    plt.title(rf"Trial collapse: $h_c={hc:.6f}$, $\nu={nu:.3f}$")
    plt.grid(alpha=0.3)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def peak_table(summary):
    """
    Report peak position and peak height for each M.
    """
    rows = []
    for M, ss in summary.groupby("M"):
        ss = ss.sort_values("h")
        imax_mean = ss["mean_Lambda"].idxmax()
        imax_med = ss["median_Lambda"].idxmax()

        r_mean = ss.loc[imax_mean]
        r_med = ss.loc[imax_med]

        rows.append({
            "M": int(M),
            "h_peak_mean": r_mean["h"],
            "Lambda_peak_mean": r_mean["mean_Lambda"],
            "std_at_peak": r_mean["std_Lambda"],
            "h_peak_median": r_med["h"],
            "Lambda_peak_median": r_med["median_Lambda"],
        })

    return pd.DataFrame(rows).sort_values("M")


# Load combined summary if needed
# summary_all = pd.read_csv("tm_left_clean_combined_summary.csv")
# raw_all = pd.read_csv("tm_left_clean_combined_raw.csv")

plot_transfer_mean(
    summary_all,
    title="Left boundary transfer-matrix scan: mean over seeds"
)

plot_transfer_median(
    summary_all,
    title="Left boundary transfer-matrix scan: median over seeds"
)

plot_seed_scatter(
    raw_all,
    title="Left boundary transfer-matrix scan: individual seed values"
)

peaks = peak_table(summary_all)
display(peaks)

# Trial collapses.
# These are only diagnostics. Do not treat them as final exponent fits.
plot_trial_collapse(summary_all, hc=-3.450, nu=1.0, use_median=True)
plot_trial_collapse(summary_all, hc=-3.450, nu=2.4, use_median=True)